# Day 1 — First API Call

## What I learned today

- The `Anthropic()` SDK auto-reads `ANTHROPIC_API_KEY` from environment 
  variables — no need to pass it explicitly
- `client.messages.create()` returns a `Message` object (Pydantic model)
- `message.content` is a **list of blocks**, not a string. For simple text 
  responses there's 1 TextBlock; presumably more blocks appear when tool use 
  is involved (Phase 2 will reveal this)
- Pricing/usage tracking is built-in via `message.usage`
- `stop_reason='end_turn'` means natural completion. Other values include
  `'max_tokens'` (truncated), `'tool_use'` (Phase 2), and `'stop_sequence'` 
- Model family: Opus (smartest, expensive) / Sonnet (balanced) / Haiku (fast, cheap)



## Design choices I made + reasoning

- **Model: `claude-haiku-4-5-20251001`** — cheapest, fast enough for 
  learning. Will reconsider in Phase 4 (model tiering).
- **max_tokens: 1000** — generous default. For "1+1=?" probably needed <20, 
  but setting higher avoids stop_reason='max_tokens' surprises during exploration.

## Day 1 self-check (the 4 questions I should be able to answer)

1. `stop_reason='end_turn'` means: Claude finished naturally. 
   Other possibilities: 'max_tokens', 'tool_use', 'stop_sequence'.
2. Input tokens > text length because: tokenization splits words into 
   subwords, and there's overhead from the message format (role, etc.).
3. If max_tokens=5: response gets cut off mid-sentence, stop_reason 
   becomes 'max_tokens'.
4. content is a list because: tool use will produce mixed blocks 
   (TextBlock + ToolUseBlock), agentic interaction is multi-block.
5. What are the 3 models in the Claude model family? What are the properties of each of them?

In [1]:
# load environment variables from a .env file into your Python process s
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# create a API client

# Note: The Anthropic SDK automatically looks for ANTHROPIC_API_KEY in the environment,
#  so after load_dotenv() you can create API client directly

from anthropic import Anthropic
client = Anthropic()

In [3]:
# making the first request 
model = "claude-haiku-4-5-20251001"
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages = [
        {
        "role" : "user",
        "content" : "1+1=?"
        }
    ]
)

In [4]:
# see full response
print(message)

Message(id='msg_018Gpbx41Q5RfVvdykfjQRZd', container=None, content=[TextBlock(citations=None, text='1 + 1 = 2', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=13, output_tokens=13, server_tool_use=None, service_tier='standard'))


In [5]:
# see message type
print(type(message))
print(dir(message))

<class 'anthropic.types.message.Message'>
['__abstractmethods__', '__annotations__', '__class__', '__class_getitem__', '__class_vars__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__fields__', '__fields_set__', '__format__', '__ge__', '__get_pydantic_core_schema__', '__get_pydantic_json_schema__', '__getattr__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__pretty__', '__private_attributes__', '__pydantic_complete__', '__pydantic_computed_fields__', '__pydantic_core_schema__', '__pydantic_custom_init__', '__pydantic_decorators__', '__pydantic_extra__', '__pydantic_extra_info__', '__pydantic_fields__', '__pydantic_fields_set__', '__pydantic_generic_metadata__', '__pydantic_init_subclass__', '__pydantic_on_complete__', '__pydantic_parent_namespace__', '__pydantic_post_init__', '__pydantic_private__', '__pydantic_root_model__', '__

In [6]:
# see messgae content 
message.content

[TextBlock(citations=None, text='1 + 1 = 2', type='text')]

In [7]:
# Extract the text
message.content[0].text

'1 + 1 = 2'

In [8]:
# Check token usage 
message.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=13, output_tokens=13, server_tool_use=None, service_tier='standard')

In [9]:
# check stop reason
message.stop_reason

'end_turn'